# IMDB Pre-processing

In [1]:
import spacy
import nltk
import glob
import random
from pathlib import Path
import tqdm
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

nlp = spacy.load('en_core_web_lg')


/home/trdp/Software/anaconda3/envs/phd_env/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [2]:
DATASET_PATH = "../data/datasets/IMDB/train/raw/@SENTIMENT/*.txt"


In [3]:
positive_imdb = glob.glob(DATASET_PATH.replace("@SENTIMENT", "positive"))
negative_imdb = glob.glob(DATASET_PATH.replace("@SENTIMENT", "negative"))

train_imdb = positive_imdb + negative_imdb

random.shuffle(train_imdb)
print(f"Files available: {len(train_imdb)}")

Files available: 40000


In [4]:
positive_raw_dataset = [open(f).read() for f in tqdm.tqdm(positive_imdb, desc="Reading positive reviews")]
negative_raw_dataset = [open(f).read() for f in tqdm.tqdm(negative_imdb, desc="Reading negative reviews")]


Reading negative reviews: 100%|██████████| 20000/20000 [00:00<00:00, 35807.13it/s]


In [5]:
# Temporary
train_raw_dataset = positive_raw_dataset + negative_raw_dataset

In [6]:
from bs4 import BeautifulSoup
import re

def preprocessing_imdb(raw_text: str) -> str:
    """
    Strips non-linguistic HTML artifacts ("true noise")
    and normalizes whitespace ("byproduct noise") to prepare
    raw text for a dependency parser like SpaCy.

    This function *preserves* all linguistic signals,
    including punctuation, casing, and stopwords.
    """

    # 1. Parse HTML to remove tags and decode entities (e.g., &nbsp;)
    # We use "html.parser" as it's built-in and sufficient.
    try:
        text_content = BeautifulSoup(raw_text, "html.parser").get_text()
    except Exception as e:
        # Fallback for any unexpected parsing error
        print(f"BeautifulSoup parsing error: {e}. Falling back to raw text.")
        text_content = raw_text

    # 2. Normalize all whitespace (spaces, \n, \t) into a single space
    # and remove any leading/trailing whitespace.
    normalized_text = re.sub(r'\s+', ' ', text_content).strip()

    return normalized_text

In [7]:


random.shuffle(train_raw_dataset)
text = train_raw_dataset[0]
print("-" * 50)
print(text)
print("-" * 50)
print(preprocessing_imdb(text))

--------------------------------------------------
Seeing as the world snooker championship final finished in a premature and disappointing manner with Ronnie O`Sullivan defeating Greame Dott by 18 frames to 8 BBC 2 found a gap in their schedule and so decided to broadcast A WALK ON THE MOON a movie I had absolutely no knowledge off<br /><br />I missed a few seconds of the title credits so had no idea Viggo Mortensen starred in it and thought possibly it might be a cheap TVM , certainly the opening with the mawkish Pearl and Marty taking their kids to a Summer camp has that sort of made for TV feel though the brightly lit ( Too brightly lit ) cinematography seemed to suggest this was a cinematic film and it wasn`t until the appearence of Viggo Mortensen as hippy guy Walker that I realised this was a cinema release , after all someone of Mortensen`s stature wouldn`t star in a TVM , I mean that`s like a legend like Robert DeNiro appearing in a straight to video film . Wait a minute , did

In [8]:
from src.data.utils import add_new_relation_to_graph
from src.data.text_graph_dataset_parsers import TextEmbedding
import re
import spacy
import torch
import logging
import networkx as nx
from tqdm import tqdm
from typing import List, Tuple, Set, Optional, Callable


class Text2DP:
    def __init__(self, lang="english", max_num_nodes=1000):
        self.spacy_models = {
            "italian": "it_core_news_lg",
            "english": "en_core_web_lg",
            "portuguese": "pt_core_news_lg",
            "portuguese_voto": "pt_core_news_lg",
        }
        self.embeddings_path = {
            "italian": "data/external/embeddings/itwiki_20180420_100d.bin",
            "english": "data/external/embeddings/enwiki_20180420_100d.bin",
            "portuguese": "data/external/embeddings/glove_legal_100.bin",
            "portuguese_voto": "data/external/embeddings/glove_legal_100.bin",
        }

        self.lang = lang

        # Load the SpaCy model
        try:
            self.nlp = spacy.load(self.spacy_models[lang])
        except IOError:
            logging.error(f"Could not load SpaCy model {self.spacy_models[lang]}.")
            logging.error("Please run: python -m spacy download {self.spacy_models[lang]}")
            raise

        self.max_num_nodes = max_num_nodes
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # Word Embeddings
        # Assumes TextEmbedding class is defined and imported
        self.text_embedding = TextEmbedding(
            model_name="glove",
            file_path=self.embeddings_path[self.lang]
        )

    def parse_corpus(self, corpus: List[str], preprocessing_fn: Optional[Callable] = None) -> List[nx.MultiDiGraph]:
        """
        Parse a corpus of text into a list of Dependency Parsing graph representations.

        Args:
            corpus (List[str]): The text corpus to parse.
            preprocessing_fn (Optional[Callable]): A (str -> str) function for preprocessing
                                                  the text before parsing. Defaults to None.
        Returns:
            List[nx.MultiDiGraph]: A list of graphs.
        """
        graphs = []
        for text in tqdm(corpus, desc="Parsing corpus"):
            graph = self.parse_document(text, preprocessing_fn=preprocessing_fn)
            if graph is not None:
                graphs.append(graph)
        return graphs

    def parse_document(self, raw_text: str, preprocessing_fn: Optional[Callable] = None) -> nx.MultiDiGraph | None:
        """
        Parse an individual document into a dependency graph,
        optionally adding sequential edges.

        Args:
            raw_text (str): The raw text document to parse.
            preprocessing_fn (Callable, optional): A (str -> str) preprocessing function.
                                                  Defaults to None.
        Returns:
            nx.MultiDiGraph: The graph representation of the parsed document.
        """
        if not raw_text or not raw_text.strip():
            logging.warning("Input text is empty or contains only whitespace. Skipping.")
            return None

        # --- PRINCIPLE 1: PREPROCESSING IS SEPARATE ---
        # Apply preprocessing (str -> str) BEFORE SpaCy parsing.
        clean_text = raw_text
        if preprocessing_fn:
            clean_text = preprocessing_fn(clean_text)

        if not clean_text or not clean_text.strip():
            logging.warning("Text is empty after preprocessing. Skipping.")
            return None

        # Apply the SpaCy pipeline
        doc = self.nlp(clean_text)

        graph = nx.MultiDiGraph()

        # --- PRINCIPLE 2: BUILD THE *FULL* GRAPH ---
        self._add_dependency_edges(graph, doc)

        # self._add_sequential_edges(graph, doc) # Still optional

        if graph.number_of_nodes() == 0:
            logging.warning("No nodes were added to the graph. Skipping.")
            return None

        if graph.number_of_nodes() >= self.max_num_nodes:
            logging.warning(
                f"Graph grew too large ({graph.number_of_nodes()} nodes > {self.max_num_nodes}). Skipping graph.")
            return None

        return graph

    def _add_dependency_edges(self, graph: nx.MultiDiGraph, doc: spacy.tokens.Doc) -> None:
        """
        Add ALL dependency parsing edges to the graph.
        Nodes are identified by their lemma.
        """
        # We track status to stop if the graph gets too big
        continue_adding = True

        for token in doc:
            if not continue_adding:
                break

            # --- PRINCIPLE 3: INCLUDE ALL TOKENS ---
            # We only skip tokens that are their own head (ROOT) or are spaces.
            # All other tokens (PUNCT, ADP, DET...) are *essential* nodes.
            if token == token.head or token.is_space:
                continue

            # Assumes add_new_relation_to_graph handles node existence
            # and returns False if max_num_nodes is hit.
            status = add_new_relation_to_graph(
                target_graph=graph,
                # --- PRINCIPLE 4: NORMALIZE NODES ---
                # Use .lemma_ as the canonical node identifier.
                node_1_lemma=token.head.lemma_,
                node_1_pos=token.head.pos_,
                node_2_lemma=token.lemma_,
                node_2_pos=token.pos_,
                edge=token.dep_,
                text_embedding=self.text_embedding,
                device=self.device,
                max_num_nodes=self.max_num_nodes
            )

            if not status:
                logging.info(f"Max nodes ({self.max_num_nodes}) hit. Stopping edge addition.")
                continue_adding = False

    def _add_sequential_edges(self, graph: nx.MultiDiGraph, doc: spacy.tokens.Doc) -> None:
        """Add sequential edges to the graph to represent the natural word order."""

        # The only valid filter is to remove non-linguistic space tokens.
        valid_tokens = [token for token in doc if not token.is_space]

        continue_adding = True
        for current_token, next_token in zip(valid_tokens[:-1], valid_tokens[1:]):
            if not continue_adding:
                break

            # --- PRINCIPLE 5: BE CONSISTENT ---
            # We MUST pass the same arguments (lemma, pos) as the dependency
            # function to ensure node attributes are consistent.
            status = add_new_relation_to_graph(
                target_graph=graph,
                node_1_lemma=current_token.lemma_,
                node_1_pos=current_token.pos_,
                node_2_lemma=next_token.lemma_,
                node_2_pos=next_token.pos_,
                edge="sequence",
                text_embedding=self.text_embedding,
                device=self.device,
                max_num_nodes=self.max_num_nodes
            )

            if not status:
                logging.info(f"Max nodes ({self.max_num_nodes}) hit. Stopping edge addition.")
                continue_adding = False

[nltk_data] Downloading package punkt to /home/trdp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
